In [2]:
# Imports
import astropy.units as u
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import scipy
from datetime import date
import time

matplotlib.use("tkagg")

In [3]:
# Define constants
sag_a_r_s = 1.2 * 10 ** 10 * u.m  # https://academic.oup.com/mnras/article/421/2/1315/1134797?login=false
r_g = sag_a_r_s / 2
c = 2.99 * 10 ** 8 * (u.m / u.s)

In [4]:
def q_squared_to_b_squared(q, v, r_s):
    """Convert from periastron to impact parameter b"""
    gamma = 1 / np.sqrt(1 - (v / c) ** 2)
    b_squared = q ** 2 * (c / v) ** 2 * (1 / (1 - r_s / q) - 1 / gamma ** 2)
    return b_squared

In [78]:
def process_time_step_orbit(r, dt, energy_term, h):
    """Calculate changes to r and the vaporization loss"""
    dr = -1 * np.sqrt(energy_term - (1 - sag_a_r_s / r) * (c ** 2 + h ** 2 / r ** 2)) * dt
    return dr

last_time = 0

def process_time_step_orbit_solve_ivp(t, r_from_q, energy_term, h, q, lightspeed, r_s):
    """Calculate changes to r and the vaporization loss, modified for solve_ivp"""
    # global last_time
    # print(t - last_time)
    # last_time = t
    r = abs(r_from_q) + q
    dr = -1 * np.sqrt(energy_term - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2))
    return dr + 0.01 


def process_radius_step_orbit_solve_ivp(r, t, energy_term, h, lightspeed, r_s):
    """Calculate the change in time for """
    d_tau = 1 / np.sqrt(energy_term - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2))
    d_t = 1 / (lightspeed * (1 - r_s / r) * np.sqrt(1 - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2) * 1 / energy_term))
    return np.array([d_tau, d_t])


def calc_radius_loss(distance, fudge, dt):
    dr_frag = fudge * 10 * u.m / (200 * u.s) * (distance / (10 * r_g)) ** -3 * dt
    return dr_frag


def check_if_correct_q(q, b, v):
    q = q * r_g
    total = ((b / r_g) ** 2 - q ** 2 * c ** 2 / v ** 2 * (1 / (1 - sag_a_r_s / q) - (1 - (v / c) ** 2)) / r_g ** 2).value
    return total

def numerically_determine_q(b, v):
    """Numerically determine the closest approach"""
    """Has issues"""
    result = scipy.optimize.minimize(check_if_correct_q, np.array([8]), args=(b, v), bounds=[(4.01, None)], method="Nelder-Mead")
    return result.x

In [ ]:
# Testing new ways to numerically determine q
actual_q = 4.001 * r_g
v = 10 ** 5 * u.m / u.s
b = np.sqrt(q_squared_to_b_squared(actual_q, v, sag_a_r_s))
new_q_start = time.perf_counter()
result_q = numerically_determine_q(b, v)
new_q_end = time.perf_counter()
print(new_q_end - new_q_start)
new_q = scipy.optimize.fsolve(check_if_correct_q, np.array([8]), args=(b, v))
print(actual_q / r_g, check_if_correct_q(actual_q / r_g, b, v))
print(result_q, actual_q / r_g - result_q, check_if_correct_q(result_q, b, v))
print(new_q, check_if_correct_q(new_q, b, v),  - actual_q / r_g + new_q)
plt.figure()
plt.plot(np.linspace(3, 30, 5000), check_if_correct_q(np.linspace(3, 30, 5000), b, v))
# plt.vlines(result_q, ymin=-10 ** -8, ymax=10 ** 8, label="minimize determined q", colors='r', linestyles="dashed")
plt.vlines(actual_q / r_g, ymin=-10 ** -8, ymax=10 ** 8, label="actual q", colors='g', linestyles="dashed")
plt.vlines(new_q, ymin=-10 ** -8, ymax=10 ** 8, label="fsolve determined q", colors='b', linestyles="dashed")
plt.xlabel("q [r_g]")
plt.ylabel("$b^2 - b(q)^2$")
plt.legend()
# plt.ylim(0, 10 ** 14)
plt.title(f"Actual q = {actual_q / r_g:.2f}r_g")
plt.show()

0.0016332860104739666
4.001 8.881784197001252e-16
[4.01] [-0.009] [775630.12245745]
[4.001] [5.68434189e-14] [1.38049572e-11]


In [77]:
v = 10 ** 5 * u.m / u.s
qs_to_test = np.linspace(4.001, 30, 1000)
results = np.zeros((1, 3))
for q in qs_to_test:
    b = np.sqrt(q_squared_to_b_squared(q * r_g, v, sag_a_r_s))
    test_start = time.perf_counter()
    fsolve_q = scipy.optimize.fsolve(check_if_correct_q, np.array([8]), args=(b, v))[0]
    test_end = time.perf_counter()
    result = np.array([q, np.sqrt((q - fsolve_q) ** 2), test_end - test_start])
    results = np.vstack((results, result))

print(np.max(results, axis=0))

/tmp/ipykernel_2405934/1565791078.py:7: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  fsolve_q = scipy.optimize.fsolve(check_if_correct_q, np.array([8]), args=(b, v))[0]


[3.00000000e+01 1.38049572e-11 9.42339585e-03]


In [6]:
def integrate_orbit(q, v_0, dt=0.1 * u.s, output=False):
    """Integrate the radius at a given time for an orbit with the given parameters."""
    steps_done = 0
    b_0 = np.sqrt(q_squared_to_b_squared(q, v_0, sag_a_r_s))
    d_0 = 30 * r_g
    orbit_energy = c ** 2 * 1 / (1 - (v_0 / c) ** 2)
    h = v_0 * b_0 * 1 / np.sqrt(1 - (v_0 / c) ** 2)
    distances = np.array([d_0.value]) * u.m
    # q = numerically_determine_q(b_0, v_0)[0] * r_g
    # if np.log10(check_if_correct_q(q / r_g, b_0, v_0)) > 50:
    #     print(f"Orbit: {b_0}, {v_0} has undefined q")
    #     return 0, 0, False
    # if q >= 30 * r_g:
    #     print(f"Orbit: {b_0}, {v_0} doesn't tidally disrupt")
    #     return 0, 0, False
    # print(check_if_correct_q(q / r_g, b_0, v_0), q / r_g)

    # Integrate the orbit to find distance at each t
    print(d_0, q)
    while d_0 > q:
        delta_r = process_time_step_orbit(d_0, dt, orbit_energy, h)
        d_0 += delta_r
        distances = np.append(distances, d_0)
        steps_done += 1
        if steps_done % 100 == 0 and output:
            print(steps_done, d_0 / r_g)

    times = np.arange(steps_done + 1) * dt
    times_since_periastron = times - times[-1]

    return distances, times_since_periastron, True

In [7]:
def simulate_store_orbit(q_0, v_0, output=False):
    """Simulate and record the integrated orbit"""
    distances, times_since_periastron, success = integrate_orbit(q_0, v_0, output=output)
    print(len(times_since_periastron))
    if not success:
        return
    orbits_completed_file = "Text_files/orbits_integrated.csv"
    integrations_file = "Text_files/integrations.csv"

    with open(orbits_completed_file, "a+") as orbits_completed:
        orbits_completed.seek(0)
        existing_orbits = orbits_completed.readlines()
        if f"{date.today()},{q_0},{v_0}\n" in existing_orbits:
            print("orbit exists")
            return

        with open(integrations_file, "a") as integrations:
            rows = [f"{date.today()},{q_0.value},{v_0.value},{t:.1f},{d:.0f}" for t, d in np.column_stack((times_since_periastron.value, distances.value))]
            text_to_write = "\n".join(rows)
            integrations.write(text_to_write + "\n")
            integrations.close()

        orbits_completed.write(f"{date.today()},{q_0},{v_0}\n")
        orbits_completed.close()


In [8]:
# Generate and record many orbits to build a database
# vs_to_test = np.logspace(0, 6, 10) * u.m / u.s
# qs_to_test = np.linspace(4.1, 30, 10) * r_g
#
# for current_v in vs_to_test:
#     for current_q in qs_to_test:
#         print(current_q / r_g, current_v)
#         simulate_store_orbit(current_q, current_v, output=False)

In [9]:
# Investigate how timestep matters
# q, v = 5 * r_g, 1770 * u.m / u.s
# dts = [1, 0.1, 0.001]
# plt.figure()
# for time_step in dts:
#     distances, times_since_periastron, success = integrate_orbit(q, v, time_step * u.s)
#     plt.plot(times_since_periastron - times_since_periastron[0], distances, label=f"{time_step}s")
#
# plt.show()

In [10]:
# Read in orbits from text file
def get_specific_orbit(q_to_get, v_to_get, orbit_file):
    """Retrieve the distances and times for a given orbit, using date, q and v"""
    all_orbits = np.loadtxt(orbit_file, delimiter=",", usecols=(1, 2, 3, 4))
    correct_orbit = all_orbits[all_orbits[:, 0] == q_to_get][all_orbits[all_orbits[:, 0] == q_to_get][:, 1] == v_to_get]
    print(f"The given orbit has {len(correct_orbit[:, 0])} matching points")
    return correct_orbit[:, 2], correct_orbit[:, 3]


# Read in all orbits from text file and format as a list of orbits
def get_all_orbits(orbit_file):
    """Get all orbits from orbit file"""
    all_orbits = np.loadtxt(orbit_file, delimiter=",", usecols=(1, 2, 3, 4))
    processed_orbits = []
    current_orbit = all_orbits[0, 0:2]
    current_orbit_data = np.array([0, 0, 0, 0])
    for row_index in range(len(all_orbits[:, 0])):
        if not np.array_equal(all_orbits[row_index, 0:2], current_orbit):
            processed_orbits.append(current_orbit_data[1:, :])
            current_orbit_data = np.array([0, 0, 0, 0])
            current_orbit = all_orbits[row_index, 0:2]

        current_orbit_data = np.vstack((current_orbit_data, all_orbits[row_index, :]))

    return processed_orbits

In [11]:
# Integrate to find the time of periastron
def eqn_to_integrate_for_peri_time(r, energy_term, h, q, lightspeed, r_s):
    """Equation that gets integrated"""
    denom = np.sqrt(energy_term - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2))
    return -1 / denom


def calc_peri_time(q, args):
    """Integrate with quadrature to analytically find the periastron time"""
    result = scipy.integrate.quad(eqn_to_integrate_for_peri_time, 60 * r_g.value, q.value, args=args)
    return result

In [12]:
# Integrate over an orbit using solve_ivp and return its luminosity curve
def integrate_luminosity(times, v_inf, q, progenitor_radius, vaporization_fudge_factor, boulder_size_power_law, perihelion_time, min_boulder_size, max_boulder_size, prints=False):

    # Other parameters
    if prints:
        print(v_inf)
    distance_from_q = 60 * r_g - q
    initial_radius = np.linspace(min_boulder_size, max_boulder_size, 1000) * u.m
    rho = 100 * u.kg / (u.m ** 3)
    epsilon = 0.1
    b_inf = np.sqrt(q_squared_to_b_squared(q, v_inf, sag_a_r_s))
    energy = c ** 2 * 1 / (1 - (v_inf / c) ** 2)
    angular_momentum_h = v_inf * b_inf * 1 / np.sqrt(1 - (v_inf / c) ** 2)

    # Integrate the orbit to find distance at each t
    # while num_steps < 4000:
    #     change_r = process_time_step_orbit(abs(distance_from_q) + q, time_step, energy, angular_momentum_h)
    #     distance_from_q += change_r
    #     radiuses = np.append(radiuses, abs(distance_from_q) + q)
    #     num_steps += 1
    #     if num_steps % 100 == 0:
    #         print(num_steps, (abs(distance_from_q) + q) / r_g)
    solve_ivp_args = (energy.value, angular_momentum_h.value, q.value, c.value, sag_a_r_s.value)
    start = time.perf_counter()
    peri_time = calc_peri_time(q, solve_ivp_args)
    end = time.perf_counter()
    if prints:
        print(peri_time, end - start)
        print("int started")
    start = time.perf_counter()
    ode_result = scipy.integrate.solve_ivp(process_time_step_orbit_solve_ivp, (0, np.max(times) + 1), [distance_from_q.value], t_eval=times, args=solve_ivp_args, rtol=10**-10, atol=3)
    end = time.perf_counter()
    if prints:
        print(end - start)
        print("int finished")

    radius_loss_at_each_t = calc_radius_loss(abs(ode_result.y[0, :]) * u.m + q, vaporization_fudge_factor, 1 * u.s)
    total_radius_lost = np.cumulative_sum(radius_loss_at_each_t)
    radiuses_at_each_time = np.transpose(np.tile(total_radius_lost, (len(initial_radius), 1)))
    radius_at_t = np.add(-1 * radiuses_at_each_time, initial_radius)
    radius_at_t[radius_at_t < 0] = 0
    luminosities = (epsilon * c.value ** 2) * 4 * np.pi * rho * radius_at_t ** 2 * np.transpose(np.tile(radius_loss_at_each_t, (len(initial_radius), 1)))
    scale_factors = initial_radius ** -boulder_size_power_law
    total_volume = (4/3) * np.pi * progenitor_radius ** 3
    unscaled_fragment_volume = np.sum(scale_factors * (4/3) * np.pi * initial_radius ** 3)
    scale_factors_volume_adjusted = scale_factors * total_volume / unscaled_fragment_volume
    scaled_luminosities = np.sum(luminosities * scale_factors_volume_adjusted[None, :], axis=1)
    scaled_luminosities[abs(ode_result.y[0, :]) * u.m + q > 30 * r_g] = 0
    return scaled_luminosities

In [119]:
# Testing the tolerances and the runtime
brightness_6 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0)
brightness_8 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0)
brightness_4 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0)
plt.figure()
plt.plot(brightness_4, label="absolute tolerance=10^-2")
plt.plot(brightness_6, label="absolute tolerance=10^-6")
plt.plot(brightness_8, label="absolute tolerance=10^-4")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Luminosity")
plt.show()

(2233.041838408241, 6.148293323349208e-07) 0.0008024000562727451
int started
0.06573089997982606
int finished
(2233.041838408241, 6.148293323349208e-07) 0.0004708000342361629
int started
0.043741999950725585
int finished
(2233.041838408241, 6.148293323349208e-07) 0.000908799993339926
int started
0.13226360001135617
int finished


In [33]:
# Experiment with different variables
min_boulders_to_test = np.linspace(100, 500, 5)
plt.figure()
for s in min_boulders_to_test:
    plt.plot(integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0, 10, 150), label=f"max_boulder={s}m")
    print(s)
plt.legend()
plt.xlabel("Time")
plt.ylabel("Luminosity")
plt.title(f"q=5r_g, v=180, r_iso=5000, slope=1.8, fudge=1, min_bor=10m")
# plt.ylim(bottom=0)
plt.show()

180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.0007878000033088028
int started
0.0406253999681212
int finished
100.0
180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.00037759996484965086
int started
0.037635000015143305
int finished
200.0
180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.0003931000246666372
int started
0.03614959999686107
int finished
300.0
180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.0003934000269509852
int started
0.03360010002506897
int finished
400.0
180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.00040499999886378646
int started
0.045041400007903576
int finished
500.0


In [13]:
def calc_error_in_flare_shape(inputs, flare_info, plot=False):
    """Calculate the error in flare shape for a given set of flare parameters"""

    # Define a constant to convert from JWST mJy to Watts
    fraction_of_luminosity_in_2_micron = 10 ** -27

    v_unitless, q_of_r_g, r_i, vap_fudge, boulder_size_power_law, perihelion_time, min_boulder_size, max_boulder_size, platform = inputs
    q = q_of_r_g * r_g
    v = v_unitless * u.m / u.s
    b_inf = np.sqrt(q_squared_to_b_squared(q, v, sag_a_r_s))
    energy = c ** 2 * 1 / (1 - (v / c) ** 2)
    angular_momentum_h = v * b_inf * 1 / np.sqrt(1 - (v / c) ** 2)
    args = (energy.value, angular_momentum_h.value, q.value, c.value, sag_a_r_s.value)
    peri_time = calc_peri_time(q, args)[0]
    obs_times = flare_info[:, 0]
    times = peri_time + (obs_times - perihelion_time)
    flare_brightness = integrate_luminosity(times, v, q, r_i, vap_fudge, boulder_size_power_law, perihelion_time, min_boulder_size, max_boulder_size, plot)
    scaled_brightness = flare_brightness * fraction_of_luminosity_in_2_micron
    if plot:
        plt.figure()
        plt.plot(times, flare_info[:, 1] - platform, label="Observed")
        plt.plot(times, scaled_brightness, label="Model")
        plt.legend()
        plt.xlabel("Time")
        plt.ylabel("Luminosity [mJy]")
        plt.show()
    error = flare_info[:, 1] - platform - scaled_brightness.value
    return np.sum(error ** 2)

In [ ]:
# Load in section of observed light curve
jwst_data = np.loadtxt('sgra_frame_lightcurve_all.csv', delimiter=',', skiprows=1, usecols=(4, 7, 8))
plt.figure()
plt.plot(jwst_data[:, 1])
plt.show()
section_around_flare = jwst_data[abs(jwst_data[:, 0] - 60405.35070) * 24 * 3600 < 400]
section_around_flare[:, 0] = (section_around_flare[:, 0] - 60405.35070) * 24 * 3600


In [63]:
guess = np.array([10 ** 5, 4.1, 3700, 1, 2, 420, 1, 200, 1.3])
calc_error_in_flare_shape(guess, section_around_flare, plot=True)

100000.0 m / s
(5571.462989673468, 3.035871577594662e-05) 0.0005385000004025642
int started
0.015652599999157246
int finished


np.float64(33.41778744235849)

In [64]:
# Minimize the error to find the optimal orbit
result = scipy.optimize.minimize(calc_error_in_flare_shape, guess, bounds=((1, 10 ** 6), (4.1, 30), (1, 1000000), (0.01, 10), (0.1, 10), (-1000, 1000), (1, 10), (100, 300), (-10, 10)), args=section_around_flare)
calc_error_in_flare_shape(result.x, section_around_flare, True)
print(result.x, calc_error_in_flare_shape(result.x, section_around_flare))

Exception ignored in: <function WeakSet.__init__.<locals>._remove at 0x0000010A2425D580>
Traceback (most recent call last):
  File "C:\Users\thkel\AppData\Local\Programs\Python\Python313\Lib\_weakrefset.py", line 39, in _remove
    def _remove(item, selfref=ref(self)):
KeyboardInterrupt: 

KeyboardInterrupt



In [65]:
result = scipy.optimize.differential_evolution(calc_error_in_flare_shape, ((1, 10 ** 6), (4.1, 30), (1, 1000000), (0.01, 10), (0.1, 10), (-1000, 1000), (1, 10), (100, 300), (-10, 10)), args=(section_around_flare,))
calc_error_in_flare_shape(result.x, section_around_flare, True)
print(result.x, calc_error_in_flare_shape(result.x, section_around_flare))

C:\Users\thkel\AppData\Local\Temp\ipykernel_23584\313026193.py:14: RuntimeWarning: invalid value encountered in sqrt
  dr = -1 * np.sqrt(energy_term - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2))


601744.5981527577 m / s
(6391.143962635566, 3.041895979549736e-08) 0.0007501000000047497
int started
0.01570259999971313
int finished
[6.01744598e+05 2.99346676e+01 7.53617979e+05 2.88727742e+00
 7.93905204e+00 8.61701875e+02 4.65908555e+00 2.76686722e+02
 1.54105074e+00] 18.33358577897863
